https://docs.ifcopenshell.org/ifccsv.html

## Structured Attribute and Property Extraction

This approach extracts all attributes and properties in a more organized way:

In [ ]:
import pandas as pd
import ifcopenshell
import ifcopenshell.util.element

# Load the model
model = ifcopenshell.open("../models/AC20-FZK-Haus.ifc")

def get_element_data(element):
    """Extract structured data from an IFC element"""
    data = {}
    
    # Basic attributes
    data['GlobalId'] = element.GlobalId
    data['Name'] = element.Name or ''
    data['Description'] = element.Description or ''
    data['ObjectType'] = element.ObjectType or ''
    data['Tag'] = element.Tag or ''
    
    # Type-specific attributes
    if hasattr(element, 'PredefinedType'):
        data['PredefinedType'] = str(element.PredefinedType) if element.PredefinedType else ''
    
    # Get all property sets
    psets = ifcopenshell.util.element.get_psets(element)
    
    # Flatten property sets into columns
    for pset_name, pset_data in psets.items():
        if isinstance(pset_data, dict):
            for prop_name, prop_value in pset_data.items():
                if prop_name != 'id':  # Skip the ID field
                    column_name = f"{pset_name}.{prop_name}"
                    data[column_name] = prop_value
    
    return data

# Get all walls
walls = model.by_type("IfcWall")

# Extract data for all walls
wall_data = [get_element_data(wall) for wall in walls]

# Create DataFrame
df_structured = pd.DataFrame(wall_data)

# Display first few rows
print(f"Total walls: {len(df_structured)}")
print(f"Total columns: {len(df_structured.columns)}")
print("\nColumn names:")
for col in df_structured.columns:
    print(f"  - {col}")

df_structured.head()

In [ ]:
# View specific property sets
print("Property Set Columns:")
pset_columns = [col for col in df_structured.columns if '.' in col]
for col in sorted(pset_columns):
    print(f"  {col}")
    
# Display DataFrame with cleaner formatting
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

# Check which columns actually exist before displaying
available_cols = ['Name']
for col in ['Pset_WallCommon.IsExternal', 'Pset_WallCommon.LoadBearing', 
            'BaseQuantities.Length', 'BaseQuantities.Height', 'BaseQuantities.Width']:
    if col in df_structured.columns:
        available_cols.append(col)

print(f"\nDisplaying columns: {available_cols}")
df_structured[available_cols].head(10)

In [ ]:
# Export to CSV with all structured data
output_path = "../data/walls_structured.csv"
df_structured.to_csv(output_path, index=False)
print(f"Exported to {output_path}")

# Try to export to Excel if openpyxl is available
try:
    excel_path = "../data/walls_structured.xlsx"
    df_structured.to_excel(excel_path, index=False, sheet_name='Walls')
    print(f"Exported to {excel_path}")
except ImportError:
    print("Note: Install openpyxl with 'uv add openpyxl' to enable Excel export")

In [ ]:
# More advanced: Get attributes and properties for multiple element types
def extract_all_elements(model, element_types=None):
    """Extract data for multiple element types"""
    
    if element_types is None:
        element_types = ['IfcWall', 'IfcSlab', 'IfcBeam', 'IfcColumn', 'IfcDoor', 'IfcWindow']
    
    all_data = []
    
    for element_type in element_types:
        elements = model.by_type(element_type)
        for element in elements:
            data = get_element_data(element)
            data['IFC_Type'] = element_type
            all_data.append(data)
    
    return pd.DataFrame(all_data)

# Extract data for multiple element types
df_all = extract_all_elements(model)

print(f"Total elements: {len(df_all)}")
print("\nElements by type:")
print(df_all['IFC_Type'].value_counts())

# Group by type and show sample
df_all.groupby('IFC_Type').head(2)